In [ ]:
#import packages

import os
import pandas as pd

#define file paths

CLIMATE_DIR = os.environ.get("CLIMATE_DIR")

file_path = os.path.join(CLIMATE_DIR, "run_project/intermediate_data/LOCA_growth_stages")
save_path = os.path.join(CLIAMTE_DIR, "run_project/intermediate_data/LOCA_stage_table")

# make sure output dir exists
os.makedirs(save_path, exist_ok=True)

all_files = [os.path.join(file_path, f) for f in os.listdir(file_path) if f.endswith("final_growth_stages.csv")]

#In step 4 we got the growth stage column for each row of data, now we need a table that just gives stage, start date and end date
#this table is needed to calculate the stress indices

# Define the stages
stage_map = {
    "booting":    ["booting"],
    "flowering":  ["flowering", "flowering_grainfill"],   # include overlap
    "grain_fill": ["grain_fill", "flowering_grainfill"],  # include overlap
}

for file in all_files:
    df = pd.read_csv(file)

    # ensure date is datetime
    df["date"] = pd.to_datetime(df["date"])

    rows = []
    for (county, year), g in df.groupby(["county", "year"], sort=True):
        for out_stage, labels in stage_map.items():
            sub = g[g["stage"].isin(labels)]
            if sub.empty:
                continue
            rows.append({
                "county": county,
                "year": int(year),
                "stage": out_stage,
                "start_date": sub["date"].min().date().isoformat(),
                "end_date": sub["date"].max().date().isoformat(),
            })

    stage_dates = (pd.DataFrame(rows)
                     .sort_values(["county", "year", "stage"])
                     .reset_index(drop=True))

    output_name = os.path.basename(file).replace("_temp_gdd_merged_final_growth_stages.csv", "") + "_stage_table.csv"
    out_csv = os.path.join(save_path, output_name)

    stage_dates.to_csv(out_csv, index=False)

    print("Saved:", out_csv)
    print(stage_dates.head(10))
